# Amended continuation 003
Read docs/revival_continuation_colab.md and the artifact-bound review before execution.

In [ ]:
from pathlib import Path
import os, sys, re, subprocess, json, shutil, time
from google.colab import drive
drive.mount('/content/drive')
GIT_REF = input('Full reviewed and pushed continuation commit SHA: ').strip()
assert re.fullmatch(r'[0-9a-fA-F]{40}', GIT_REF)
REPO_URL = 'https://github.com/Soqoro/Pact.git'
SCRATCH = Path('/content/pact-continuation')
CHECKOUT = SCRATCH / ('checkout-' + GIT_REF[:12])
RUN_ID = 'pact-heterogeneous-receiver-training-continuation-003'
RUN = SCRATCH / 'runs' / RUN_ID
CACHE = SCRATCH / 'cache'
PERSISTENT = Path('/content/drive/MyDrive/PACT/revival') / RUN_ID
PARENT = Path('/content/drive/MyDrive/PACT/revival/pact-heterogeneous-receiver-training-001/bundles/pact-heterogeneous-receiver-training-001-PRIVATE-1792171769996477145.zip')
FORENSIC = Path('/content/drive/MyDrive/PACT/revival/pact-heterogeneous-receiver-training-replacement-002/forensics/replacement-002-FORENSIC-PRIVATE-1792568005277432071.zip')
REVIEW_HASH = '2546efe226d2f6d80d0b13bb04930a482487ea7477a892ec821f763de618f0db'
SCRATCH.mkdir(parents=True, exist_ok=True)
if not CHECKOUT.exists():
    subprocess.run(['git','clone',REPO_URL,str(CHECKOUT)],check=True)
assert subprocess.check_output(['git','-C',str(CHECKOUT),'remote','get-url','origin'],text=True).strip() == REPO_URL
assert not subprocess.check_output(['git','-C',str(CHECKOUT),'status','--porcelain'],text=True).strip(), 'Preserve local changes.'
subprocess.run(['git','-C',str(CHECKOUT),'fetch','origin',GIT_REF],check=True)
subprocess.run(['git','-C',str(CHECKOUT),'checkout','--detach',GIT_REF],check=True)
assert subprocess.check_output(['git','-C',str(CHECKOUT),'rev-parse','HEAD'],text=True).strip().lower() == GIT_REF.lower()
os.chdir(CHECKOUT)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-e','.[model]'],check=True)
from google.colab import userdata
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
assert os.environ.get('HF_TOKEN')
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '0'
sys.path.insert(0,str(CHECKOUT/'src'))
subprocess.run([sys.executable,'-c',"import torch; from transformers import AutoTokenizer, AutoModelForCausalLM; assert torch.cuda.is_available() and torch.cuda.device_count()==1 and torch.cuda.is_bf16_supported(); print(torch.__version__, torch.cuda.get_device_name(0))"],check=True)


In [ ]:
from pact.util import read_json, file_hash, digest, redact

def study(stage, *options):
    cmd = [sys.executable,'-u','-m','pact.revival.continuation',stage,
           '--run-dir',str(RUN),'--persistent',str(PERSISTENT),
           '--cache-dir',str(CACHE),*map(str,options)]
    log = SCRATCH / f'continuation-{stage}-{time.time_ns()}.log'
    print('Starting:',stage,'Log:',log,flush=True)
    with log.open('w') as output:
        proc = subprocess.Popen(cmd,stdout=output,stderr=subprocess.STDOUT)
    offset = 0
    last = time.monotonic()
    while True:
        with log.open(errors='replace') as stream:
            stream.seek(offset)
            text = stream.read()
            offset = stream.tell()
        if text:
            print(redact(text),end='',flush=True)
            last = time.monotonic()
        if proc.poll() is not None:
            with log.open(errors='replace') as stream:
                stream.seek(offset)
                print(redact(stream.read()),end='',flush=True)
            break
        if time.monotonic()-last >= 30:
            print(stage,'still running; log:',log,flush=True)
            last = time.monotonic()
        time.sleep(1)
    if proc.returncode:
        raise RuntimeError(f'{stage} failed ({proc.returncode}); preserve files and log. No automatic retry.')
    print('Finished:',stage,flush=True)


In [ ]:
study('review')
if RUN.exists():
    assert (RUN/'plan.json').is_file(), 'Partial preparation: stop and preserve it.'
    study('inspect')
elif PERSISTENT.exists() and any(PERSISTENT.iterdir()):
    study('restore')
else:
    sources = SCRATCH/'sources'
    sources.mkdir(parents=True,exist_ok=True)
    for source, name, expected in (
        (PARENT,'parent-private.zip','f9a78141d0c662c06be903b49318f0235f6253d11e42f5bb17b2dd8adf9f2793'),
        (FORENSIC,'forensic-private.zip','bb9d759af00b2ee7942ad25236bced0b6ebf95cbdaccf2f582a22586148d9015'),
    ):
        assert source.is_file(), str(source)
        target = sources/name
        shutil.copyfile(source,target)
        assert file_hash(target)==expected, f'Checksum mismatch: {name}'
    study('prepare','--parent-private',sources/'parent-private.zip',
          '--forensic',sources/'forensic-private.zip','--review-hash',REVIEW_HASH)
PLAN_HASH = digest(read_json(RUN/'plan.json'))
print('Continuation plan hash:',PLAN_HASH)
study('inspect')


In [ ]:
study('repair-tokenizer-cache','--execute','--plan-hash',PLAN_HASH)
study('preflight','--execute','--plan-hash',PLAN_HASH)


In [ ]:
status = read_json(RUN/'training/uniform_joint_sft/Q/status.json')
if status['completed_steps'] == 93:
    study('train-agent','--family','Q','--resume','--stop-after','94',
          '--execute','--plan-hash',PLAN_HASH)
study('report')
assert read_json(RUN/'summary.json')['recovery_safe']
study('export')


In [ ]:
for family in ('Q','L','M'):
    folder = RUN/'training/uniform_joint_sft'/family
    status = read_json(folder/'status.json') if (folder/'status.json').exists() else {}
    if status.get('complete'):
        print('Already complete:',family,flush=True)
        continue
    options = ['--family',family,'--execute','--plan-hash',PLAN_HASH]
    if (folder/'contract.json').exists():
        options.append('--resume')
    study('train-agent',*options)
    study('report')


In [ ]:
study('report')
summary = read_json(RUN/'summary.json')
print(json.dumps(summary,indent=2))
assert summary['training_complete'] and summary['recovery_safe']
assert summary['new_work']['committed_updates']==291
assert summary['combined_committed_updates']==768
assert summary['new_work']['generation_calls']==0
study('export')
print('Return the PRIVATE and SANITIZED bundles with printed SHA256 values.')
print('Keep RECOVERY on Drive. Original step 94 remains historically unresolved.')
